# <font color="Green">**Notebook Purpose**</font>

This notebook implements a sensitivity analysis to address **Reviewer 1's Comment 4**, which asks for robustness checks on key modeling decisions — specifically the choice of six-month binning. The original analysis grouped each patient's prescription history into 12 half-year bins between 2019 and 2024. This notebook rebuilds the same trajectories at a **three-month (quarterly) resolution** — 24 bins per patient — then re-runs Ward's linkage clustering at k=40 and compares the resulting therapy-group assignments to the original.

Structurally, this notebook mirrors `SensitivityAnalysis_SparsityFilter.ipynb`. The only substantive differences are:
1. Instead of loading pre-computed `patient_vectors.pkl`, we rebuild trajectories from `medication_info.csv` with 3-month bins (replicating the logic in `MedicationTrajectoryRepresentations.ipynb`, changing only the binning function and bin count).
2. Because the new feature space is 24×10 = 240-dim (vs. the original 12×10 = 120-dim), centroid-based cluster-to-group matching uses the **original 6-month `patient_bins.pkl`** to compute profiles of the new clusters — this keeps the centroids comparable to the original 108-dim cluster profiles.
3. No patient-level sparsity filter is applied here, so all 18,652 patients (including original Early Dropout) appear in the concordance matrix.

---

### <font color="Red">Required Data</font>

1. **`medication_info.csv`** — raw medication table with `patient_id`, `medication_class`, `start_date`. Used to rebuild trajectories at 3-month resolution.
2. **`patient_demographics.csv`** — loaded for parity with the original pipeline.
3. **`patient_bins.pkl`** — the original **6-month** bins (from `MedicationTrajectoryRepresentations.ipynb`). Used for centroid matching in Section 6.
4. **Original group pkl files** (from `ClusteringAnalysis.ipynb`):
   - `monotherapy_patients.pkl`
   - `dual_therapy_patients.pkl`
   - `complex_therapy_patients.pkl`
   - `GLP_1_therapy_patients.pkl`
   - `variant_therapy_patients.pkl`
   - `early_dropout_patients.pkl`
   - `sorted_cluster_patient_ids.pkl` — maps original cluster ID → list of patient IDs (needed for original cluster centroids).


## Section 1 — Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import pickle

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Raw data for trajectory rebuild
patient_demographics = pd.read_csv('/content/patient_demographics.csv')
medication_info = pd.read_csv('/content/medication_info.csv')

# Original 6-month bins — needed for centroid matching in Section 6
with open('/content/patient_bins.pkl', 'rb') as f:
    patient_bins_6m = pickle.load(f)

# Original group assignments
with open('/content/monotherapy_patients.pkl', 'rb') as f:
    monotherapy_patients = pickle.load(f)

with open('/content/dual_therapy_patients.pkl', 'rb') as f:
    dual_therapy_patients = pickle.load(f)

with open('/content/complex_therapy_patients.pkl', 'rb') as f:
    complex_therapy_patients = pickle.load(f)

with open('/content/GLP_1_therapy_patients.pkl', 'rb') as f:
    GLP_1_therapy_patients = pickle.load(f)

with open('/content/variant_therapy_patients.pkl', 'rb') as f:
    variant_therapy_patients = pickle.load(f)

with open('/content/early_dropout_patients.pkl', 'rb') as f:
    early_dropout_patients = pickle.load(f)

with open('/content/sorted_cluster_patient_ids.pkl', 'rb') as f:
    sorted_cluster_patient_ids = pickle.load(f)

print(f'Medication records: {len(medication_info):,}')
print(f'Unique patients in medication_info: {medication_info["patient_id"].nunique():,}')
print(f'Patients in original 6-month patient_bins: {len(patient_bins_6m):,}')
print(f'Original clusters: {len(sorted_cluster_patient_ids)}')

In [ ]:
# Build original group lookup: patient_id -> group name
original_group_lookup = {}

group_dicts = {
    'Monotherapy': monotherapy_patients,
    'Dual Therapy': dual_therapy_patients,
    'Complex Therapy': complex_therapy_patients,
    'GLP-1 Therapy': GLP_1_therapy_patients,
    'Variant Therapy': variant_therapy_patients,
    'Early Dropout': early_dropout_patients,
}

for group_name, cluster_dict in group_dicts.items():
    for cluster_id, patient_list in cluster_dict.items():
        for pid in patient_list:
            original_group_lookup[pid] = group_name

# Build original CLUSTER -> group mapping (for centroid matching)
original_cluster_group = {}
for group_name, cluster_dict in group_dicts.items():
    for cluster_id in cluster_dict.keys():
        original_cluster_group[cluster_id] = group_name

print(f'Patients with an original group label: {len(original_group_lookup):,}')
print(f'Original cluster-to-group mappings: {len(original_cluster_group)}')
for g in ['Monotherapy', 'Dual Therapy', 'Complex Therapy', 'GLP-1 Therapy', 'Variant Therapy', 'Early Dropout']:
    ids = sorted([k for k,v in original_cluster_group.items() if v == g])
    print(f'  {g}: clusters {ids}')

## Section 2 — Rebuild Trajectories at 3-Month Resolution

This section mirrors `MedicationTrajectoryRepresentations.ipynb` exactly, with two changes:
- The binning function maps each date to one of **24** quarter-year bins (Q1 2019 = 0, Q4 2024 = 23) instead of 12 half-year bins.
- Loop ranges and vector construction are updated from 12 → 24 bins.

All other logic is preserved byte-for-byte: sorting by `start_date`, `{'nothing'}` imputation, the `med_class_to_index` dictionary, the `vector = [0] * 10` per-bin slot, and the flatten step. This yields a **240-dim** trajectory vector per patient (24 bins × 10 slots/bin).

### Creating `patient_bins_3m`

In [ ]:
# 3-month bin indexer: maps any date in 2019-2024 to an index in [0, 23].
# Q1 2019 -> 0, Q2 2019 -> 1, ..., Q4 2024 -> 23.
def get_quarter_index(date):
    return (date.year - 2019) * 4 + ((date.month - 1) // 3)

NUM_BINS_3M = 24

In [ ]:
medication_info['start_date'] = pd.to_datetime(medication_info['start_date'])

patient_bins_3m = {}

for patient_id, patient_info in medication_info.groupby('patient_id'):

    patient_info = patient_info.sort_values(by='start_date')

    for _, row in patient_info.iterrows():

        medication_class = row['medication_class']

        start_date = row['start_date']

        quarter_index = get_quarter_index(start_date)


        if 0 <= quarter_index < NUM_BINS_3M:

          if patient_id not in patient_bins_3m:
              patient_bins_3m[patient_id] = [set() for _ in range(NUM_BINS_3M)]

          patient_bins_3m[patient_id][quarter_index].add(medication_class)

        else:
            print(f"Invalid quarter index: {quarter_index}")

Impute `'nothing'` into the empty sets.

In [ ]:
for patient_id, bins in patient_bins_3m.items():
    for i, bin in enumerate(bins):
        if not bin:
            bins[i] = {'nothing'}

print(f'Patients with 3-month trajectories: {len(patient_bins_3m):,}')

Example patient from `patient_bins_3m` (24 quarterly bins):

In [ ]:
unique_ids = medication_info['patient_id'].unique()
random_patient_id = np.random.choice(unique_ids)

print('Patient id: ', random_patient_id)
patient_bins_3m[random_patient_id]

### Creating `patient_vectors_3m`

Same one-hot encoding scheme as `MedicationTrajectoryRepresentations.ipynb` — 10-slot per-bin vector (9 medication classes + 1 unused slot, preserved from the original code), concatenated across all 24 bins into a **240-dimensional vector** per patient.

**Medication-class indices (unchanged from original):**

- **MET:** 0
- **SUL:** 1
- **SGLT2:** 2
- **GLP-1:** 3
- **GIP/GLP-1:** 4
- **Insulin:** 5
- **DPP-4:** 6
- **TZD:** 7
- **Other:** 8

In [ ]:
med_class_to_index = {
    'MET': 0,
    'SUL': 1,
    'SGLT2': 2,
    'GLP-1': 3,
    'GIP/GLP-1': 4,
    'Insulin': 5,
    'DPP-4': 6,
    'TZD': 7,
    'Other': 8
}

patient_vectors_3m = {}

for patient_id, bins in patient_bins_3m.items():
    trajectory_vectors = []

    for bin_set in bins:
        vector = [0] * 10
        for med_class in bin_set:
            # Skip the 'nothing' placeholder inserted by the imputation step above;
            # it's not an unknown class, and an all-zero bin-vector is the desired encoding.
            if med_class == 'nothing':
                continue
            if med_class in med_class_to_index:
                vector[med_class_to_index[med_class]] = 1
            else:
                print(f"Unknown medication class: {med_class}")
        trajectory_vectors.append(vector)

    # Flatten the 24 x 10 matrix into a 240-dimensional vector
    patient_vectors_3m[patient_id] = np.array(trajectory_vectors).flatten()

print(f'Patients vectorized: {len(patient_vectors_3m):,}')
print(f'Vector dimensionality: {len(next(iter(patient_vectors_3m.values())))}')

In [ ]:
# Sanity check: compare bins and vector for a random patient
random_patient_id = np.random.choice(unique_ids)

print('Patient id: ', random_patient_id)
print('3-month bins:')
print(patient_bins_3m[random_patient_id])
print()
print('240-dim vector:')
print(patient_vectors_3m[random_patient_id])

## Section 3 — Re-Clustering

Ward's linkage agglomerative clustering at k=40, matching the primary analysis and the sparsity sensitivity analysis. All 18,652 patients are clustered — no sparsity filter is applied in this sensitivity analysis.

In [ ]:
# Build matrix for clustering
filtered_patient_ids = sorted(patient_vectors_3m.keys())
X_3m = np.array([patient_vectors_3m[pid] for pid in filtered_patient_ids])

print(f'Clustering {len(filtered_patient_ids):,} patients with {X_3m.shape[1]} features')

clustering = AgglomerativeClustering(
    n_clusters=40,
    linkage='ward',
    metric='euclidean'
)

clustering.fit(X_3m)
print('Clustering complete.')

In [ ]:
# Build cluster dictionaries (sorted by size, largest = cluster 1)
cluster_results = pd.DataFrame({
    'patient_id': filtered_patient_ids,
    'cluster': clustering.labels_
})

cluster_patient_ids_raw = cluster_results.groupby('cluster')['patient_id'].apply(list).to_dict()

sorted_clusters = sorted(cluster_patient_ids_raw.items(), key=lambda x: len(x[1]), reverse=True)

new_cluster_patient_ids = {}
for i, (_, patients) in enumerate(sorted_clusters):
    new_cluster_patient_ids[i + 1] = patients

print(f'Created {len(new_cluster_patient_ids)} clusters.')
print(f'Cluster sizes: min={min(len(v) for v in new_cluster_patient_ids.values())}, '
      f'max={max(len(v) for v in new_cluster_patient_ids.values())}, '
      f'median={np.median([len(v) for v in new_cluster_patient_ids.values()]):.0f}')

## Section 4 — Cluster Summary Table

Quick overview of each new cluster: size and dominant drug classes in the first 3-month bin (2019 Q1). Use this as a cheat sheet when reviewing plots and therapy-group assignments.

In [ ]:
def get_top_drugs_bin1(patient_ids, patient_bins_src, top_n=3):
    """Return top drug classes by prevalence in bin 0 (2019 Q1)."""
    drug_counts = {}
    for pid in patient_ids:
        for drug in patient_bins_src[pid][0]:
            if drug != 'nothing':
                drug_counts[drug] = drug_counts.get(drug, 0) + 1
    total = len(patient_ids)
    sorted_drugs = sorted(drug_counts.items(), key=lambda x: x[1], reverse=True)
    return ', '.join(f'{d} ({c/total*100:.0f}%)' for d, c in sorted_drugs[:top_n])


summary_rows = []
for cid in sorted(new_cluster_patient_ids.keys()):
    pids = new_cluster_patient_ids[cid]
    top_drugs = get_top_drugs_bin1(pids, patient_bins_3m)
    summary_rows.append({
        'Cluster': cid,
        'N Patients': len(pids),
        'Top Drugs (2019 Q1)': top_drugs if top_drugs else 'none'
    })

cluster_summary_df = pd.DataFrame(summary_rows)
cluster_summary_df

## Section 5 — Visualization Functions & Cluster Viewer

Two-panel visualizations for each cluster: (1) patient-level medication heatmap, (2) medication distribution line chart. These are adapted from `ClusterLevelPlotsCreation.ipynb` / `SensitivityAnalysis_SparsityFilter.ipynb` Section 5, with the x-axis updated to 24 quarterly bins:

- **Major ticks / labels**: year (2019–2024), placed at each year's first quarter.
- **Minor ticks**: one per quarter (24 total), unlabeled.

In [ ]:
PRESCRIPTION_COLORS = {
    'SGLT2': '#00FF00',
    'SUL': '#FFB6C1',
    'Insulin': '#FF0000',
    'MET': '#0000FF',
    'DPP-4': '#8B4513',
    'GLP-1': '#FFDB58',
    'GIP/GLP-1': '#40E0D0',
    'TZD': '#FF8C00',
    'Other': '#000000',
    'nothing': '#FFFFFF'
}

def mix_colors(colors):
    rgb_colors = np.array([to_rgb(PRESCRIPTION_COLORS[color])
                           for color in colors if color in PRESCRIPTION_COLORS])
    if len(rgb_colors) == 0:
        return np.array(to_rgb(PRESCRIPTION_COLORS['nothing']))
    return np.mean(rgb_colors, axis=0).astype(np.float32)


def _apply_quarterly_xaxis(ax):
    """Year labels at each year's first quarter; unlabeled minor ticks for the other quarters."""
    year_start_positions = [0, 4, 8, 12, 16, 20]  # Q1 of 2019..2024
    year_labels = [str(y) for y in range(2019, 2025)]
    ax.set_xticks(year_start_positions)
    ax.set_xticklabels(year_labels, rotation=45, ha='right')
    ax.set_xticks(range(NUM_BINS_3M), minor=True)
    ax.tick_params(axis='x', which='minor', length=3)
    ax.tick_params(axis='x', which='major', length=6)


def plot_cluster_heatmap(ax, cluster_patient_ids, patient_bins_cluster):
    heatmap_data = np.ones((len(cluster_patient_ids), NUM_BINS_3M, 3), dtype=np.float32)
    for i, patient_id in enumerate(cluster_patient_ids):
        for j, prescriptions in enumerate(patient_bins_cluster[patient_id]):
            heatmap_data[i, j] = mix_colors(prescriptions)
    ax.imshow(heatmap_data, aspect='auto', interpolation='none')
    _apply_quarterly_xaxis(ax)
    ax.set_yticks([])
    ax.set_title('1. Individual Medication Sequences', fontsize=14)


def plot_medication_distribution(ax, cluster_patient_ids, patient_bins_cluster):
    drug_classes = [d for d in PRESCRIPTION_COLORS.keys() if d != 'nothing']
    drug_counts = {drug: [0] * NUM_BINS_3M for drug in drug_classes}
    total_patients = len(cluster_patient_ids)
    for patient_id in cluster_patient_ids:
        for bin_index, prescriptions in enumerate(patient_bins_cluster[patient_id]):
            for drug in prescriptions:
                if drug in drug_counts:
                    drug_counts[drug][bin_index] += 1
    for drug in drug_counts:
        drug_counts[drug] = [count / total_patients * 100 for count in drug_counts[drug]]
    for drug, percentages in drug_counts.items():
        ax.plot(range(NUM_BINS_3M), percentages, label=drug,
                color=PRESCRIPTION_COLORS[drug], linewidth=2)
    _apply_quarterly_xaxis(ax)
    ax.set_ylabel('% of Patients on Medication', fontsize=11)
    ax.set_title('2. Medication Class Distribution Over Time', fontsize=14)
    ax.set_ylim(0, 100)
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)
    ax.grid(True, alpha=0.3)

In [ ]:
def view_cluster(cluster_id):
    """Display the 2-panel visualization for a given cluster."""
    if cluster_id not in new_cluster_patient_ids:
        print(f'Cluster {cluster_id} not found. Valid IDs: {sorted(new_cluster_patient_ids.keys())}')
        return

    pids = new_cluster_patient_ids[cluster_id]
    patient_bins_cluster = {pid: patient_bins_3m[pid] for pid in pids}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    plot_cluster_heatmap(axes[0], pids, patient_bins_cluster)
    plot_medication_distribution(axes[1], pids, patient_bins_cluster)

    fig.suptitle(f'3-Month Sensitivity Analysis — Cluster {cluster_id} (n={len(pids):,})',
                 fontsize=16, fontweight='bold')
    fig.subplots_adjust(left=0.05, right=0.97, top=0.85, bottom=0.12, wspace=0.18)
    plt.show()


# Example: view the largest cluster
view_cluster(1)

In [ ]:
# View all clusters sequentially (uncomment to run)
# for cid in sorted(new_cluster_patient_ids.keys()):
#     view_cluster(cid)

## Section 6 — Automated Therapy Group Assignment via Cluster-Level Centroid Matching

Same approach as `SensitivityAnalysis_SparsityFilter.ipynb` Section 6, with two adaptations:

1. **Feature space for centroids**: both original-cluster and new-cluster centroids are computed from the **original 6-month `patient_bins_6m`**. This gives 108-dim vectors on both sides (12 bins × 9 drug classes) so cosine similarity is directly meaningful. Because every new cluster contains the same patients as some subset of original clusters, each patient still has a valid 6-month profile.
2. **Early Dropout is included**. In the sparsity notebook, dropout clusters were excluded from the candidate pool because the sparsity filter removed most dropout patients. Here, all 18,652 patients are retained, so dropout clusters are part of the candidate pool and new clusters can be assigned to the Early Dropout group when their profile matches.

The pipeline:

1. Compute a medication-profile centroid for each of the **40 original clusters** (108-dim vectors over 12 bins × 9 classes, using 6-month bins).
2. Compute the same type of centroid for each of the **40 new clusters** (also using the 6-month bins, grouping by the new cluster membership).
3. For each new cluster, find its nearest original cluster by cosine similarity and assign the new cluster to that original cluster's therapy group (one of the 6 groups including Early Dropout).
4. Flag assignments where the runner-up group is within a 0.05 similarity margin for optional manual review.

In [ ]:
DRUG_CLASSES = ['MET', 'SUL', 'SGLT2', 'GLP-1', 'GIP/GLP-1', 'Insulin', 'DPP-4', 'TZD', 'Other']
NUM_BINS_6M = 12
CONFIDENCE_MARGIN = 0.05  # flag if top two groups are within this margin

def compute_cluster_profile(patient_ids, patient_bins_src):
    """
    Compute the medication profile centroid for a cluster using 6-month bins.
    Returns a 108-dim vector: for each of 12 bins, the proportion of
    patients taking each of 9 drug classes.
    """
    n = len(patient_ids)
    if n == 0:
        return np.zeros(NUM_BINS_6M * len(DRUG_CLASSES))

    profile = np.zeros((NUM_BINS_6M, len(DRUG_CLASSES)))
    for pid in patient_ids:
        bins = patient_bins_src[pid]
        for bin_idx, bin_set in enumerate(bins):
            for drug_idx, drug in enumerate(DRUG_CLASSES):
                if drug in bin_set:
                    profile[bin_idx, drug_idx] += 1

    profile = profile / n  # convert counts to proportions
    return profile.flatten()

In [ ]:
# Step 1: Compute centroids for ALL 40 original clusters (including Early Dropout)
original_centroids = {}
for cid in sorted(original_cluster_group.keys()):
    pids = sorted_cluster_patient_ids[cid]
    original_centroids[cid] = compute_cluster_profile(pids, patient_bins_6m)

print(f'Computed centroids for {len(original_centroids)} original clusters')
print(f'Groups represented: {sorted(set(original_cluster_group[c] for c in original_centroids))}')

In [ ]:
# Step 2: Compute centroids for each NEW cluster, using 6-month bins
# (same patients, just grouped by new cluster membership)
new_centroids = {}
for cid, pids in new_cluster_patient_ids.items():
    new_centroids[cid] = compute_cluster_profile(pids, patient_bins_6m)

print(f'Computed centroids for {len(new_centroids)} new clusters (using 6-month bins)')

In [ ]:
# Step 3: For each new cluster, find nearest original cluster + per-group best matches

orig_cids = sorted(original_centroids.keys())
orig_matrix = np.array([original_centroids[cid] for cid in orig_cids])

new_cids = sorted(new_centroids.keys())
new_matrix = np.array([new_centroids[cid] for cid in new_cids])

# Cosine similarity: (n_new x n_orig)
sim_matrix = cosine_similarity(new_matrix, orig_matrix)

assignment_records = []

for i, new_cid in enumerate(new_cids):
    sims = sim_matrix[i]

    # Best overall match
    best_idx = np.argmax(sims)
    best_orig_cid = orig_cids[best_idx]
    best_sim = sims[best_idx]
    assigned_group = original_cluster_group[best_orig_cid]

    # Best match PER GROUP (to see runner-up from a different group) —
    # includes Early Dropout since dropout clusters are part of the candidate pool
    group_best = {}
    for j, orig_cid in enumerate(orig_cids):
        g = original_cluster_group[orig_cid]
        if g not in group_best or sims[j] > group_best[g]['sim']:
            group_best[g] = {'orig_cid': orig_cid, 'sim': sims[j]}

    # Runner-up: best match from a DIFFERENT group
    runner_up_group = None
    runner_up_sim = -1
    for g, info in group_best.items():
        if g != assigned_group and info['sim'] > runner_up_sim:
            runner_up_group = g
            runner_up_sim = info['sim']

    margin = best_sim - runner_up_sim
    confident = margin > CONFIDENCE_MARGIN

    assignment_records.append({
        'New Cluster': new_cid,
        'N Patients': len(new_cluster_patient_ids[new_cid]),
        'Assigned Group': assigned_group,
        'Best Match (Orig Cluster)': best_orig_cid,
        'Similarity': round(best_sim, 4),
        'Runner-Up Group': runner_up_group,
        'Runner-Up Sim': round(runner_up_sim, 4),
        'Margin': round(margin, 4),
        'Confident': confident
    })

In [ ]:
# Step 4: Display the full assignment table
assignment_df = pd.DataFrame(assignment_records)

# Summary stats
n_confident = assignment_df['Confident'].sum()
n_flagged = len(assignment_df) - n_confident

print(f'Auto-assigned (confident, margin > {CONFIDENCE_MARGIN}): {n_confident} / {len(assignment_df)}')
print(f'Flagged for review: {n_flagged}')
print()

# Show group sizes
print('Assigned group sizes:')
group_sizes = assignment_df.groupby('Assigned Group')['N Patients'].sum()
for g in ['Monotherapy', 'Dual Therapy', 'Complex Therapy', 'GLP-1 Therapy', 'Variant Therapy', 'Early Dropout']:
    if g in group_sizes.index:
        n_clusters = (assignment_df['Assigned Group'] == g).sum()
        print(f'  {g}: {group_sizes[g]:,} patients across {n_clusters} clusters')
print()

assignment_df

In [ ]:
# Step 5: Show flagged clusters (low confidence) for optional manual review
flagged = assignment_df[~assignment_df['Confident']].copy()

if len(flagged) == 0:
    print('No clusters flagged — all assignments are confident.')
else:
    print(f'{len(flagged)} cluster(s) flagged for review (margin <= {CONFIDENCE_MARGIN}):')
    print()
    print(flagged[['New Cluster', 'N Patients', 'Assigned Group', 'Similarity',
                   'Runner-Up Group', 'Runner-Up Sim', 'Margin']].to_string(index=False))
    print()
    print('Use view_cluster(cluster_id) above to inspect these clusters visually.')
    print('To override an assignment, modify the overrides dict below and re-run.')

In [ ]:
# Inspect a flagged cluster (edit the ID to whichever you want to look at)
view_cluster(30)

### Step 5b — Cluster Membership Diagnostic

After running the assignment, a useful sanity check is to look at each new cluster's *original* group composition. The centroid match tells us which original group a new cluster looks most like; this diagnostic tells us where its members actually came from. Mismatches between the two are exactly the kind of artifact we discussed for the 3-month sensitivity: a cluster of original Monotherapy patients with long (180-day) refill cycles can look mechanically sparse in the 3-month representation and end up matched to Early Dropout, even though its members were never originally classified as dropouts.

The cells below build a per-new-cluster membership table and then surface the new clusters most likely to be such artifacts — those currently assigned to **Early Dropout** but composed largely of original **Monotherapy** patients. Use these to decide which clusters (if any) to override before re-running Section 7.

In [ ]:
GROUP_ORDER = sorted(set(original_group_lookup.values()))

In [ ]:
# Build a per-new-cluster table showing what fraction of each cluster's members
# came from each original group.
membership_records = []
for _, row in assignment_df.iterrows():
    new_cid = row['New Cluster']
    pids = new_cluster_patient_ids[new_cid]

    orig_counts = {g: 0 for g in GROUP_ORDER}
    for pid in pids:
        orig = original_group_lookup.get(pid)
        if orig is not None:
            orig_counts[orig] += 1

    n = len(pids)
    record = {
        'New Cluster': new_cid,
        'Assigned Group': row['Assigned Group'],
        'N Patients': n,
        'Similarity': row['Similarity'],
    }
    for g in GROUP_ORDER:
        record[f'% {g}'] = round(orig_counts[g] / n * 100, 1) if n > 0 else 0.0

    membership_records.append(record)

membership_df = pd.DataFrame(membership_records)
membership_df

In [ ]:
# Focus on new clusters currently assigned to Early Dropout, ranked by the share of
# their members that were originally Monotherapy. High % Monotherapy + low % Early
# Dropout is the artifact pattern: these clusters are sparse-looking in the 3-month
# representation but their members aren't real dropouts.

early_dropout_assignments = (
    membership_df[membership_df['Assigned Group'] == 'Early Dropout']
    .sort_values('% Monotherapy', ascending=False)
    .reset_index(drop=True)
)

print(f'{len(early_dropout_assignments)} new clusters currently assigned to Early Dropout.')
print(f'Total patients in these clusters: {early_dropout_assignments["N Patients"].sum():,}')
print()

display_cols = ['New Cluster', 'N Patients', 'Similarity'] + [f'% {g}' for g in GROUP_ORDER]
early_dropout_assignments[display_cols]

In [ ]:
# Visual inspection of each Early Dropout-assigned cluster.
# Each call to view_cluster() produces the standard 2-panel figure (heatmap + line chart).
# Comment out the loop and use view_cluster(<cid>) directly if you want to inspect one at a time.

for _, row in early_dropout_assignments.iterrows():
    cid = row['New Cluster']
    print(f'\n=== Cluster {cid} | n={row["N Patients"]} | '
          f'{row["% Monotherapy"]}% orig. Monotherapy | '
          f'{row["% Early Dropout"]}% orig. Early Dropout | '
          f'similarity={row["Similarity"]:.3f} ===')
    view_cluster(cid)

In [ ]:
# Step 6: Optional manual overrides for flagged clusters
# After reviewing flagged clusters with view_cluster(), add overrides here.
# Example: overrides = {14: 'Complex Therapy', 27: 'Variant Therapy'}

overrides = {8: "Monotherapy", 25: "Monotherapy", 26: "Dual Therapy", 34: "Dual Therapy"}  # <-- fill in if needed after reviewing flagged clusters

# Apply overrides
for cid, group in overrides.items():
    mask = assignment_df['New Cluster'] == cid
    old_group = assignment_df.loc[mask, 'Assigned Group'].values[0]
    assignment_df.loc[mask, 'Assigned Group'] = group
    print(f'Override: Cluster {cid} changed from {old_group} -> {group}')

if not overrides:
    print('No overrides applied. Using all automated assignments.')

In [ ]:
# Build the final group lookup and group assignment dicts (same format as original notebook)
GROUP_ORDER = ['Monotherapy', 'Dual Therapy', 'Complex Therapy', 'GLP-1 Therapy', 'Variant Therapy', 'Early Dropout']

new_group_assignments = {g: [] for g in GROUP_ORDER}
for _, row in assignment_df.iterrows():
    new_group_assignments[row['Assigned Group']].append(row['New Cluster'])

new_group_lookup = {}
for group_name, cluster_ids in new_group_assignments.items():
    for cid in cluster_ids:
        for pid in new_cluster_patient_ids[cid]:
            new_group_lookup[pid] = group_name

# Print final group sizes
print('Final therapy group sizes:')
for group in GROUP_ORDER:
    n = sum(1 for v in new_group_lookup.values() if v == group)
    cids = sorted(new_group_assignments[group])
    print(f'  {group}: {n:,} patients | clusters: {cids}')

## Section 7 — Concordance / Confusion Matrix

Cross-tabulate original vs. new therapy-group assignments. Because this sensitivity analysis applies no patient-level filter, the overlap pool is **all 18,652 patients** — the Early Dropout group is therefore included as a row and column in the confusion matrix, as discussed.

In [ ]:
# Identify the overlap (should be the full 18,652 with no filtering)
overlap_pids = set(original_group_lookup.keys()) & set(new_group_lookup.keys())

print(f'Patients with original group label: {len(original_group_lookup):,}')
print(f'Patients in new clustering:         {len(new_group_lookup):,}')
print(f'Overlap (in both):                  {len(overlap_pids):,}')

In [ ]:
# Build the confusion matrix for overlap patients — includes Early Dropout
records = []
for pid in overlap_pids:
    records.append({
        'patient_id': pid,
        'Original Group': original_group_lookup[pid],
        'New Group': new_group_lookup[pid]
    })

comparison_df = pd.DataFrame(records)

confusion = pd.crosstab(
    comparison_df['Original Group'],
    comparison_df['New Group'],
    margins=True,
    margins_name='Total'
)

# Reorder rows and columns
row_order = [g for g in GROUP_ORDER if g in confusion.index] + ['Total']
col_order = [g for g in GROUP_ORDER if g in confusion.columns] + ['Total']
confusion = confusion.reindex(index=row_order, columns=col_order, fill_value=0)

print('Concordance Matrix (Original rows × New columns):')
print()
confusion

In [ ]:
# Concordance rate per group (diagonal / row total)
print('Concordance rates (% of original group retained in same new group):')
print()
for group in GROUP_ORDER:
    if group in confusion.index and group in confusion.columns:
        diagonal = confusion.loc[group, group]
        row_total = confusion.loc[group, 'Total']
        rate = diagonal / row_total * 100 if row_total > 0 else 0
        print(f'  {group:20s}: {diagonal:,} / {row_total:,} = {rate:.1f}%')

In [ ]:
import matplotlib as mpl
from pathlib import Path

# --- Journal-quality global settings (set once per session; safe to re-set) ---
mpl.rcParams['pdf.fonttype']       = 42       # TrueType, not Type 3
mpl.rcParams['ps.fonttype']        = 42
mpl.rcParams['font.family']        = 'sans-serif'
mpl.rcParams['font.sans-serif']    = ['Arial', 'Helvetica', 'DejaVu Sans']
mpl.rcParams['savefig.bbox']       = 'tight'
mpl.rcParams['savefig.pad_inches'] = 0.05

# --- Heatmap of concordance percentages — includes Early Dropout ---
# Order rows/columns so the diagonal reads: Early Dropout, GLP-1, Mono, Dual, Variant, Complex
CONCORDANCE_ORDER = ['Early Dropout', 'GLP-1 Therapy', 'Monotherapy', 'Dual Therapy', 'Variant Therapy', 'Complex Therapy']

# Drop any group that ended up empty (e.g., via override) so .loc doesn't throw
present = [g for g in CONCORDANCE_ORDER if g in confusion.index and g in confusion.columns]

confusion_core = confusion.loc[present, present].copy()
confusion_pct = confusion_core.div(confusion_core.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 6))

sns.heatmap(confusion_pct, annot=True, fmt='.1f', cmap='Blues', ax=ax,
            vmin=0, vmax=100, annot_kws={'size': 11})

ax.set_title('Patient Retention Across Therapy Groups (%) — 3-Month Binning Sensitivity',
             fontsize=13, fontweight='bold', pad=18)
ax.set_xlabel('3-Month Binning Group Assignment',
              fontsize=12, labelpad=10)
ax.set_ylabel('Original (6-Month) Group Assignment',
              fontsize=12, labelpad=10)

ax.tick_params(axis='both', labelsize=10)

plt.tight_layout()

# --- Export in multiple formats ---
out_dir = Path('/content/figures/supplement')
out_dir.mkdir(parents=True, exist_ok=True)
stem = 'efig12_3month_concordance'

fig.savefig(out_dir / f'{stem}.pdf')              # vector — primary submission file
fig.savefig(out_dir / f'{stem}.png', dpi=600)     # raster — 600 dpi fallback

plt.show()
print(f'Saved {stem}.pdf and {stem}.png to {out_dir}/')

## Section 8 — Export

Export new clustering results and group assignments for downstream use. File names are prefixed `sensitivity_3m_` to distinguish them from the sparsity sensitivity outputs.

In [ ]:
# Export new cluster-patient dictionary
with open('/content/sensitivity_3m_cluster_patient_ids.pkl', 'wb') as f:
    pickle.dump(new_cluster_patient_ids, f)

# Export new 3-month patient bins and vectors (for reproducibility)
with open('/content/sensitivity_3m_patient_bins.pkl', 'wb') as f:
    pickle.dump(patient_bins_3m, f)

with open('/content/sensitivity_3m_patient_vectors.pkl', 'wb') as f:
    pickle.dump(patient_vectors_3m, f)

# Export new group dictionaries (including Early Dropout)
for group_name, cluster_ids in new_group_assignments.items():
    group_dict = {}
    for cid in cluster_ids:
        group_dict[cid] = new_cluster_patient_ids[cid]
    safe_name = group_name.lower().replace(' ', '_').replace('-', '')
    with open(f'/content/sensitivity_3m_{safe_name}_patients.pkl', 'wb') as f:
        pickle.dump(group_dict, f)
    print(f'Exported sensitivity_3m_{safe_name}_patients.pkl')

# Export confusion matrix
confusion.to_excel('/content/sensitivity_3m_concordance_matrix.xlsx')
print('Exported sensitivity_3m_concordance_matrix.xlsx')

## Section 9 — Key Numbers for the Response Letter

Summary of the statistics to cite when responding to Reviewer 1 Comment 4 on 3-month binning:

- Total patients clustered: [X]
- Number of new clusters (k): 40
- Vector dimensionality: 240 (24 bins × 10 slots) vs. original 120 (12 × 10)
- Automated group assignment: [X] of 40 clusters assigned confidently (cosine similarity margin > 0.05), [X] flagged for review
- Concordance rates by group:
  - Monotherapy: 70.6%
  - Dual Therapy: 52.5%
  - Complex Therapy: 8.5%
  - GLP-1 Therapy: 72.3%
  - Variant Therapy: 9.9%
  - Early Dropout: 85.5%

**Suggested language for the response letter / supplement:**

"As a sensitivity analysis, we re-built each patient's medication trajectory at three-month resolution (24 quarterly bins, 2019–2024) and re-clustered using the same Ward's linkage procedure with k=40. All 18,652 patients were included; no sparsity filter was applied. New clusters were assigned to the six therapy groups (including Early Dropout) using a nearest-centroid classifier based on cosine similarity to the original expert-curated cluster profiles, evaluated in the shared 108-dimensional six-month feature space. [X] of 40 assignments were made with high confidence (similarity margin > 0.05), and [X] were reviewed manually. Concordance rates with the primary (six-month) analysis ranged from [X]% to [X]% across the six groups, confirming that the therapy-group taxonomy is robust to the choice of binning interval."